# Day 7 — Build GPT from Scratch II: Decode & Sampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day07-build-gpt-from-scratch-ii.ipynb)

Training writes the logits; **sampling decides the words**. Today you implement the entire sampler — temperature, top-k, top-p — unit-test it against the packet's worked numbers, then drive a real model with it and compare greedy vs nucleus output.

In [ ]:
!pip install -q torch transformers --index-url https://download.pytorch.org/whl/cpu
# Expected: installs quietly, no errors.

## 1. The sampler: logits → temperature → truncate → renormalize → draw

This is the whole implementation behind every `temperature`/`top_p` API parameter. ~30 lines, no dependencies beyond torch.

In [ ]:
import torch
import torch.nn.functional as F

def sample(logits, temperature=1.0, top_k=0, top_p=1.0, seed=None):
    """Draw one token id from logits. logits: 1-D tensor of vocab scores."""
    if seed is not None:
        torch.manual_seed(seed)
    x = logits / max(temperature, 1e-9)          # 1. temperature warp
    if top_k > 0:                                 # 2a. top-k truncation
        vals, _ = torch.topk(x, top_k)
        x = torch.where(x < vals[-1], torch.tensor(float("-inf")), x)
    probs = F.softmax(x, dim=-1)
    if top_p < 1.0:                               # 2b. nucleus truncation
        srt, idx = torch.sort(probs, descending=True)
        cumsum = torch.cumsum(srt, dim=-1)
        keep = torch.cat([torch.tensor([True]),
                          cumsum[:-1] < top_p])   # smallest set with cumsum >= p
        mask = torch.zeros_like(probs, dtype=torch.bool).scatter(0, idx, keep)
        probs = torch.where(mask, probs, torch.tensor(0.0))
        probs = probs / probs.sum()               # 3. renormalize
    return torch.multinomial(probs, num_samples=1).item(), probs
# Expected: no output — just the definition.

## 2. Unit tests: the packet's worked examples, asserted

Logits `[3, 2, 1, 0]`. Temperature table from the packet: T=0.5 → `[0.865, 0.117, 0.016, 0.002]`; T=1.0 → `[0.644, 0.237, 0.087, 0.032]`; T=2.0 → `[0.426, 0.259, 0.157, 0.095]`. And top-p=0.9 at T=1.0 must give `[0.665, 0.245, 0.090]` on the surviving 3 tokens.

In [ ]:
logits = torch.tensor([3.0, 2.0, 1.0, 0.0])
expected = {0.5: [0.865, 0.117, 0.016, 0.002],
            1.0: [0.644, 0.237, 0.087, 0.032],
            2.0: [0.426, 0.259, 0.157, 0.095]}
for T, exp in expected.items():
    _, p = sample(logits, temperature=T, seed=0)
    assert torch.allclose(p, torch.tensor(exp), atol=1e-3), (T, p)
    print(f"T={T}: {[round(float(v), 3) for v in p]}  OK")

_, p = sample(logits, temperature=1.0, top_p=0.9, seed=0)
nz = p[p > 0]
assert torch.allclose(nz.sort(descending=True).values,
                      torch.tensor([0.665, 0.245, 0.090]), atol=1e-3), p
print("top-p=0.9 survivors:", [round(float(v), 3) for v in nz.sort(descending=True).values], " OK")
# Expected: three "OK" lines for T, then the top-p line.

In [ ]:
# Determinism: sampling is the ONLY nondeterministic part of inference.
r = torch.randn(5000)
a, _ = sample(r, temperature=1.0, top_p=0.9, seed=42)
b, _ = sample(r, temperature=1.0, top_p=0.9, seed=42)
c, _ = sample(r, temperature=1.0, top_p=0.9, seed=7)
assert a == b, "same seed must reproduce"
print(f"seed 42 -> {a}, seed 42 again -> {b}, seed 7 -> {c}")
print("deterministic under fixed seed: OK")
# Expected: a == b printed, c usually different.

## 3. Why top-p beats top-k: flat vs sharp distributions

Top-k is a fixed window on a moving target. Watch it fail on a flat distribution (keeps junk) while top-p adapts.

In [ ]:
flat = torch.tensor([1.0]*50 + [0.0]*50)   # model is uncertain
sharp = torch.tensor([10.0] + [0.0]*99)            # model is certain
for name, lg in (("flat ", flat), ("sharp", sharp)):
    _, pk = sample(lg, top_k=50, seed=0)
    _, pp = sample(lg, top_k=0, top_p=0.9, seed=0)
    print(f"{name}: top-k=50 keeps {(pk>0).sum().item():3d} tokens | "
          f"top-p=0.9 keeps {(pp>0).sum().item():3d} tokens")
# Expected: flat -> top-k keeps 50, top-p keeps ~45 (all live mass);
#           sharp -> top-k keeps 50 (49 junk), top-p keeps 1.

## 4. Drive a real model: distilGPT-2 with your sampler

Manual decode loop — one forward pass per token, your `sample()` choosing each one. ~40 tokens × 3 settings on CPU takes a couple of minutes. Watch coherence trade against diversity.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2").eval()
print("params:", sum(p.numel() for p in model.parameters()))
# Expected: params: 81912576

In [ ]:
@torch.no_grad()
def generate(prompt, max_new=40, **kw):
    ids = tok(prompt, return_tensors="pt").input_ids
    for _ in range(max_new):
        nxt, _ = sample(model(ids).logits[0, -1], **kw)
        ids = torch.cat([ids, torch.tensor([[nxt]])], dim=1)
        if nxt == tok.eos_token_id:
            break
    return tok.decode(ids[0])

prompt = "The king said to his advisors"
for name, kw in (("greedy      ", dict(temperature=1e-9)),
                 ("T=0.8 top-p ", dict(temperature=0.8, top_p=0.9, seed=1)),
                 ("T=1.5       ", dict(temperature=1.5, seed=1))):
    print(f"--- {name} ---")
    print(generate(prompt, **kw), "\n")
# Expected: greedy is grammatical but loops/repeats; T=0.8 is the sweet spot;
# T=1.5 wanders. Paste all three into your notes with a one-line verdict each.

## 5. Checkpoint + README pattern

Your Day 6 nanoGPT checkpoint is `gpt-shakespeare.pt`. The pattern is always the same — weights plus config, so the model is reproducible from the file alone. Demo on a tiny throwaway model (saving distilGPT-2's 353 MB would be silly).

In [ ]:
import torch.nn as nn
toy = nn.Sequential(nn.Embedding(65, 64), nn.Linear(64, 65))  # stand-in for your GPT
cfg = {"arch": "toy-gpt", "vocab": 65, "n_embd": 64, "note": "pattern demo only"}
torch.save({"model": toy.state_dict(), "config": cfg}, "/tmp/gpt-shakespeare-demo.pt")
ckpt = torch.load("/tmp/gpt-shakespeare-demo.pt", weights_only=False)
print("config:", ckpt["config"])
print("tensors:", len(ckpt["model"]), "| file restores losslessly: OK")
# Expected: config dict printed; tensors: 3.

## 6. What to measure

| Metric | Your number |
|---|---|
| top-p=0.9 survivors on [3,2,1,0] | [0.665, 0.245, 0.090] |
| Greedy output: first repeated phrase | |
| T=0.8, top-p=0.9: coherent? diverse? | |
| T=1.5: coherent? diverse? | |
| Same-seed regeneration identical? | yes |
| Your `gpt-shakespeare.pt` size (MB) | |

**Exit check:** you can predict the temperature table from memory, explain the greedy repetition loop mechanically, and your sampler passes all asserts above. Tomorrow: the KV cache — every decode step drops from O(n) to O(1).